# 23_Image 모델 학습하기

## 학습목표 
- 1. 주어진 데이터와 커스텀 데이터셋 클래스를 기반으로 모델을 학습시킵니다.
- 2. 모델 학습 과정에서 필요한 부수적 요소(데이터 로그를 기록할 수 있는 라이브러리)들을 활용하여 훈련 기록을 저장합니다.   

In [ ]:
#학습률을 시각화하는 보조 라이브러리리
!pip install tqdm

In [2]:
import os #로컬 컴퓨터에서 데이터셋을 불러오기 위해
import cv2 # 시각화 해서 표현하기 위함
import numpy as np #이미지 픽셀 데이터를 numpy 형태로 표현함
import json #개별 라벨을 읽어들어오게 함

import torch #훈련
from torch.utils.data import Dataset, DataLoader #커스텀 데이터셋을 만들기 위함
from torchvision import transforms #torchvision -> 딥러닝(이미지) 수행하는 클래스 이름 #transform 이미지를 조절
from PIL import Image #-> 이미지를 표현할 수 있는 파이썬 라이브러리 
import matplotlib.pyplot as plt
import itertools

### 데이터셋 세팅

In [29]:
class CustomData(Dataset):
    def __init__(self, image_list, label_list, transform):
        self.image_path = self.read_file_lines(image_list)
        self.label_path = self.read_file_lines(label_list)
        self.transform = transforms.Compose([
                        #전처리의 다양한 종류를 여기서 적용해준다.
                        transforms.Resize((224, 224)),
                        transforms.ToTensor(),
                        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
                    ])
        
    def __len__(self):
        #if (이미지수 == 라벨수) return 이미지 수
        return len(self.image_path)

    #텐서 형태로 변환된 이미지를 돌려줌
    def __getitem__(self, idx):
        image = self.image_path[idx]
        image = Image.open(image)    #해당 경로에서 이미지를 읽어 옴

        if self.transform is not None:
            image = self.transform(image)

        box = self.get_label_data(self.label_path[idx])

        #타겟 딕셔너리 생성
        target = {
            'boxes' : box,
            #박스의 갯수만큼 라벨을 출력해주어야 함. 우리는 클래스는 하나이므로 그냥 활용해도 됨.
            'labels' : torch.zeros(len(box), dtype=torch.int64),
            'image_id' : torch.tensor([idx], dtype=torch.int64)
        }
        print(f'image type:{type(image)}, {image}, {image.shape}')
        print(f'box:{box}, labels:{torch.zeros(len(box), dtype=torch.int64)}, image_id : {torch.tensor([idx], dtype=torch.int64)}')
        
        return image, target

    def get_label_data(self, lab):
        bounding_boxes = []
        with open(lab, 'r') as f:
            annotations = json.load(f)
            
        for i in range(len(annotations['tooth'])):
            if annotations['tooth'][i]['decayed'] == True:
                # x, y 좌표를 각각 추출
                # p[0] 기준으로 오름차순 정렬
                sorted_xs = sorted(annotations['tooth'][i]['segmentation'], key=lambda p: p[0])
                
                start_x = sorted_xs[0][0] 
                end_x = sorted_xs[-1][0]
                
                sorted_ys = sorted(annotations['tooth'][i]['segmentation'], key=lambda p: p[1])
                start_y = sorted_ys[0][1] 
                end_y = sorted_ys[-1][1]

                #x_min, y_min, x_max, y_max
                bounding_boxes.append([start_x, start_y, end_x, end_y])

        return bounding_boxes

    def read_file_lines(self, file_path):
        with open(file_path, "r", encoding="utf-8") as f:
            lines = [line.strip() for line in f]  # 개행 문자 제거 후 리스트로 변환
        return lines

In [5]:
def custom_collate_fn(batch):
    images = []
    targets = []

    for image, target in batch:
        images.append(image)  # 이미지 리스트
        targets.append(target)  # 딕셔너리 형태의 target 리스트

    return images, targets  # 리스트 형태 그대로 반환

### 훈련을 위한 디바이스 세팅

In [4]:
# train on the GPU or on the CPU, if a GPU is not available
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print(f"DEVICE : {device}")

DEVICE : cuda


### 데이터셋 세팅

In [6]:
train_label_list = 'C:/Users/jeong/Desktop/최종코드/Datasets/Proj_1/train_list.txt'
train_image_list = 'C:/Users/jeong/Desktop/최종코드/Datasets/Proj_1/train_image_list.txt'

In [7]:
valid_label_list = 'C:/Users/jeong/Desktop/최종코드/Datasets/Proj_1/valid_list.txt'
valid_image_list = 'C:/Users/jeong/Desktop/최종코드/Datasets/Proj_1/valid_image_list.txt'

In [30]:
train_dataset = CustomData(train_image_list, train_label_list, transform) #이미지에 대한 경로, 일괄적으로 적용할 전처리
train_dataloader = DataLoader(dataset = train_dataset,
                        batch_size = 2,
                        shuffle = True,
                        drop_last = False,
                        collate_fn=custom_collate_fn)

valid_dataset = CustomData(valid_image_list , valid_label_list, transform) #이미지에 대한 경로, 일괄적으로 적용할 전처리
valid_dataloader = DataLoader(dataset = valid_dataset,
                        batch_size = 2,
                        shuffle = True,
                        drop_last = False,
                        collate_fn=custom_collate_fn)

### 훈련하기

In [18]:
import torch
import torchsummary
import torchvision
import torchvision.models.detection

from tqdm import tqdm  # 학습 진행률 시각화
from torch import nn
import torchvision.transforms.functional as F

# 모델 로드 (사전학습된 Faster R-CNN)
model = torchvision.models.detection.fasterrcnn_resnet50_fpn(pretrained=True)
model.train()  # 학습 모드 설정

FasterRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=(800,), max_size=1333, mode='bilinear')
  )
  (backbone): BackboneWithFPN(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): FrozenBatchNorm2d(64, eps=0.0)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): FrozenBatchNorm2d(64, eps=0.0)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): FrozenBatchNorm2d(64, eps=0.0)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): FrozenBatchNorm2d(256, eps=0.0)
          (relu): ReLU(

In [ ]:
for epoch in range(num_epochs):
    #model.train()  # 모델 학습 모드 설정
    model.to(device)
    epoch_loss = 0

    progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{num_epochs}")  # 진행률 표시

    for images, targets in progress_bar:
        # ✅ 리스트 안의 각 이미지만 `.to(device)`로 변환
        images = [img.to(device) for img in images]
        
        # ✅ 타겟 딕셔너리 속의 텐서들도 `.to(device)` 변환
        new_targets = []
        for t in targets:
            new_target = {
                "boxes": torch.tensor(t['boxes'], dtype=torch.float32).to(device),
                "labels": torch.tensor(t['labels'], dtype=torch.int64).to(device),
                "image_id": torch.tensor(t['image_id'], dtype=torch.int64).to(device)
            }
            new_targets.append(new_target)

        print(f"======== {new_targets}========")

        # ✅ 옵티마이저 초기화
        optimizer.zero_grad()

        # ✅ 모델 학습 (Faster R-CNN은 학습 시 `loss` 반환)
        loss_dict = model(images, new_targets)
        loss = sum(loss for loss in loss_dict.values())  # 여러 개의 손실 값을 합산

        # ✅ 역전파 및 최적화
        loss.backward()
        optimizer.step()

        # ✅ 손실 값 저장
        epoch_loss += loss.item()
        progress_bar.set_postfix(loss=loss.item())

    # ✅ 에포크별 손실 출력
    print(f"Epoch [{epoch+1}/{num_epochs}] Loss: {epoch_loss:.4f}")

# ✅ 학습 완료 후 모델 저장
torch.save(model.state_dict(), "fasterrcnn_trained.pth")
print("✅ 모델 학습 완료 및 저장 완료!")

Epoch 1/10:   0%|                                                                                              | 0/11 [00:00<?, ?it/s]C:\Users\jeong\AppData\Local\Temp\ipykernel_25136\1609181489.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "labels": torch.tensor(t['labels'], dtype=torch.int64).to(device),
C:\Users\jeong\AppData\Local\Temp\ipykernel_25136\1609181489.py:18: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "image_id": torch.tensor(t['image_id'], dtype=torch.int64).to(device)


image type:<class 'torch.Tensor'>, tensor([[[-2.0665, -2.0665, -2.0665,  ..., -0.7308, -0.6965, -0.6452],
         [-2.0665, -2.0665, -2.0665,  ..., -0.6623, -0.6281, -0.5082],
         [-2.0494, -2.0494, -2.0665,  ..., -0.5767, -0.5424, -0.5424],
         ...,
         [-0.7479, -0.6965, -0.4911,  ..., -0.4226, -0.4397, -0.5767],
         [-0.7993, -0.6794, -0.4911,  ..., -0.4397, -0.4739, -0.5938],
         [-0.7993, -0.6623, -0.4739,  ..., -0.4397, -0.4739, -0.6109]],

        [[-1.9832, -1.9832, -1.9832,  ..., -1.0903, -1.0378, -0.9153],
         [-1.9832, -1.9832, -1.9832,  ..., -0.9853, -0.9328, -0.7752],
         [-1.9657, -1.9657, -1.9832,  ..., -0.8452, -0.7752, -0.7752],
         ...,
         [-1.5980, -1.5105, -1.1429,  ..., -0.9678, -0.9678, -1.0553],
         [-1.6506, -1.4755, -1.1078,  ..., -0.9503, -0.9853, -1.0728],
         [-1.6506, -1.4580, -1.0903,  ..., -0.9678, -1.0203, -1.1429]],

        [[-1.7522, -1.7522, -1.7522,  ..., -1.0376, -0.9853, -0.8807],
         [

Epoch 1/10:   9%|██████▊                                                                    | 1/11 [00:01<00:13,  1.31s/it, loss=85.2]

image type:<class 'torch.Tensor'>, tensor([[[ 0.2453,  0.2282,  0.2282,  ..., -1.2103, -1.2274, -1.2788],
         [ 0.3994,  0.3652,  0.3481,  ..., -1.1247, -1.1589, -1.1932],
         [ 0.3823,  0.3481,  0.3309,  ..., -1.1075, -1.1589, -1.1932],
         ...,
         [-0.4911, -0.5596, -0.5596,  ..., -0.5082, -0.6281, -0.6965],
         [-0.4911, -0.5253, -0.5082,  ..., -0.4739, -0.6109, -0.6794],
         [-0.5767, -0.5596, -0.4739,  ..., -0.5082, -0.6452, -0.7479]],

        [[ 0.0826,  0.0301,  0.0126,  ..., -1.3529, -1.3704, -1.4230],
         [ 0.2052,  0.1352,  0.1176,  ..., -1.3179, -1.3354, -1.3704],
         [ 0.1702,  0.1001,  0.1001,  ..., -1.3004, -1.3529, -1.3704],
         ...,
         [-0.4601, -0.5651, -0.6352,  ..., -0.8452, -1.1253, -1.3704],
         [-0.4776, -0.4776, -0.4951,  ..., -0.8102, -1.0903, -1.3704],
         [-0.5651, -0.4951, -0.3901,  ..., -0.8277, -1.0728, -1.4230]],

        [[ 0.1302,  0.0605,  0.0256,  ..., -1.2293, -1.2990, -1.3164],
         [

Epoch 1/10:  18%|█████████████▋                                                             | 2/11 [00:01<00:06,  1.35it/s, loss=81.4]

...

image type:<class 'torch.Tensor'>, tensor([[[ 1.1015,  1.0844,  1.0673,  ...,  0.8276,  0.8961,  0.9988],
         [ 1.2043,  1.1529,  1.1358,  ...,  0.9474,  0.9817,  1.0844],
         [ 1.1700,  1.1187,  1.1187,  ...,  1.0331,  1.0331,  1.0844],
         ...,
         [-0.0458, -0.1486, -0.2856,  ..., -0.8164,  0.6221,  1.3413],
         [-0.0458, -0.1828, -0.3198,  ..., -0.8164,  0.5878,  1.3242],
         [-0.1314, -0.2856, -0.4568,  ..., -0.8849,  0.3823,  1.1015]],

        [[ 0.1702,  0.1352,  0.1001,  ...,  0.2927,  0.3277,  0.3978],
         [ 0.2402,  0.1877,  0.1176,  ...,  0.3803,  0.4153,  0.4153],
         [ 0.1877,  0.1352,  0.1352,  ...,  0.4853,  0.4153,  0.3978],
         ...,
         [-1.1078, -1.1779, -1.2479,  ..., -1.5105, -0.2500,  0.4853],
         [-1.0728, -1.1779, -1.2829,  ..., -1.5280, -0.3200,  0.4678],
         [-1.1078, -1.2304, -1.3354,  ..., -1.5630, -0.4601,  0.3102]],

        [[ 0.3045,  0.2871,  0.2348,  ...,  0.2348,  0.2348,  0.2348],
         [

Epoch 10/10:  82%|████████████████████████████████████████████████████████████▌             | 9/11 [00:02<00:00,  3.10it/s, loss=41.8]

image type:<class 'torch.Tensor'>, tensor([[[-0.4226, -0.4226, -0.4397,  ..., -1.6555, -1.2959, -1.1760],
         [-0.4226, -0.4054, -0.4226,  ..., -1.5870, -1.3130, -1.0904],
         [-0.4054, -0.4054, -0.4226,  ..., -1.6727, -1.6042, -1.4329],
         ...,
         [-0.5938, -0.6965, -0.7822,  ..., -0.1486, -0.1657, -0.1143],
         [-0.5596, -0.6623, -0.7822,  ..., -0.1657, -0.1657, -0.1486],
         [-0.5082, -0.6623, -0.7993,  ..., -0.1657, -0.1486, -0.1657]],

        [[-0.0749, -0.0924, -0.0924,  ..., -1.5280, -1.1604, -1.0203],
         [-0.0749, -0.0399, -0.0749,  ..., -1.4580, -1.1604, -0.9328],
         [-0.0574, -0.0224, -0.0399,  ..., -1.5805, -1.5105, -1.3179],
         ...,
         [-1.2654, -1.3704, -1.4405,  ..., -0.7052, -0.6527, -0.6702],
         [-1.2304, -1.3354, -1.4055,  ..., -0.7402, -0.6877, -0.6702],
         [-1.1779, -1.3179, -1.3880,  ..., -0.7227, -0.6527, -0.5826]],

        [[ 0.7228,  0.7228,  0.7054,  ..., -1.1596, -0.7413, -0.5844],
         [

Epoch 10/10: 100%|█████████████████████████████████████████████████████████████████████████| 11/11 [00:03<00:00,  3.22it/s, loss=16.2]


image type:<class 'torch.Tensor'>, tensor([[[ 0.4508,  0.3652,  0.5193,  ..., -0.5424, -0.5082, -0.5253],
         [ 0.2967,  0.3994,  1.0331,  ..., -0.5082, -0.4911, -0.4911],
         [ 0.2453,  0.6563,  1.7694,  ..., -0.5596, -0.4568, -0.4739],
         ...,
         [-0.1828, -0.0458,  0.1083,  ..., -0.5424, -0.6109, -0.6794],
         [-0.1828, -0.0458,  0.0227,  ..., -0.5938, -0.5767, -0.6281],
         [-0.1999, -0.0801, -0.0116,  ..., -0.5767, -0.5424, -0.6109]],

        [[ 0.7304,  0.5903,  0.7479,  ..., -0.8452, -0.8277, -0.8452],
         [ 0.6078,  0.6254,  1.2731,  ..., -0.8277, -0.7927, -0.7927],
         [ 0.5553,  0.9055,  1.9559,  ..., -0.8627, -0.7752, -0.7927],
         ...,
         [-0.4076, -0.3375, -0.1450,  ..., -0.8452, -0.9503, -1.0203],
         [-0.4601, -0.3375, -0.2500,  ..., -0.9153, -0.8627, -0.9503],
         [-0.5126, -0.3725, -0.2675,  ..., -0.8452, -0.8277, -0.8978]],

        [[ 1.2805,  1.1062,  1.2805,  ..., -0.7238, -0.7238, -0.7238],
         [

### 로그 남기기

In [ ]:
wandb.login()

In [ ]:
wandb.login()

# 5. 모델 학습
epochs = 1000  # 학습 epoch 수
losses = []  # 손실 기록

#학습에 사용할 설정
LEARNING_RATE = 0.01
ARCHITECTURE = 'CNN'
DATASET = 'YOLO'
EPOCHS = epochs

#메타데이터 만들기
wandb.init(
  # Set the project where this run will be logged
  project="basic-intro",
  # We pass a run name (otherwise it’ll be randomly assigned, like sunshine-lollypop-10)
  name=f"experiment_{model}",
  # Track hyperparameters and run metadata
  config={
  "learning_rate": LEARNING_RATE,
  "architecture": ARCHITECTURE,
  "dataset": DATASET,
  "epochs": EPOCHS,
  })

for epoch in range(epochs):
    # 1) 순전파
    y_pred = model(X_train)

    # 2) 손실 계산
    loss = criterion(y_pred, y_train)

    # 3) 기울기 초기화
    optimizer.zero_grad()

    # 4) 역전파
    loss.backward()
    # 5) 가중치 업데이트
    optimizer.step()

    # 손실 기록
    losses.append(loss.item())
    # 2️. Log metrics from your script to W&B
    wandb.log({"loss": loss})

    if (epoch + 1) % 100 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

# Mark the run as finished
wandb.finish()